## 2. `finally` Block

The `finally` block runs **no matter what** - whether an exception was raised, caught, re-raised, or never occurred. Even `return`, `break`, or `continue` inside the `try` block will not skip `finally`.

```python
try:
    # risky code
except SomeError:
    # handle error
else:
    # no error
finally:
    # ALWAYS runs — cleanup goes here
```

### Primary use-cases for `finally`
| Resource | What to close/release |
|----------|----------------------|
| Files | `file.close()` |
| Database connections | `conn.close()` |
| Network sockets | `socket.close()` |
| Locks / mutexes | `lock.release()` |
| GUI progress dialogs | `dialog.close()` |

> 💡 In practice, the **`with` statement** (context manager) handles most resource cleanup automatically and is cleaner than `try/finally`. You'll see it everywhere for file I/O.

In [1]:
# finally always runs — even when no exception
def attempt_operation(value):
    print(f"--- attempt_operation({value!r}) ---")
    try:
        result = 100 / value
        print(f"  Result: {result}")
    except ZeroDivisionError:
        print("  Error: division by zero")
    except TypeError:
        print("  Error: invalid type")
    finally:
        print("  [finally] Cleanup complete.")   # ALWAYS prints

attempt_operation(4)
attempt_operation(0)
attempt_operation("x")

--- attempt_operation(4) ---
  Result: 25.0
  [finally] Cleanup complete.
--- attempt_operation(0) ---
  Error: division by zero
  [finally] Cleanup complete.
--- attempt_operation('x') ---
  Error: invalid type
  [finally] Cleanup complete.


In [2]:
# finally runs even when return is inside try
def read_config(path):
    print(f"Opening {path!r}...")
    try:
        f = open(path)
        data = f.read()
        return data          # return doesn't skip finally!
    except FileNotFoundError:
        print("  File not found, using defaults.")
        return {}
    finally:
        print("  [finally] Would close file handle here.")

read_config("nonexistent.cfg")

Opening 'nonexistent.cfg'...
  File not found, using defaults.
  [finally] Would close file handle here.


{}

In [3]:
# Simulated database connection pattern
class FakeDB:
    def __init__(self, name):
        self.name = name
        self.connected = False
    def connect(self):
        self.connected = True
        print(f"  [DB] Connected to '{self.name}'")
    def query(self, sql):
        if "DROP" in sql.upper():
            raise PermissionError("DROP not allowed!")
        return f"[results of: {sql}]"
    def close(self):
        self.connected = False
        print(f"  [DB] Connection to '{self.name}' closed")

def run_query(sql):
    db = FakeDB("production")
    try:
        db.connect()
        result = db.query(sql)
        print(f"  Query result: {result}")
    except PermissionError as e:
        print(f"  Permission denied: {e}")
    finally:
        db.close()    # guaranteed to run — no leaked connections

run_query("SELECT * FROM users")
print()
run_query("DROP TABLE users")

  [DB] Connected to 'production'
  Query result: [results of: SELECT * FROM users]
  [DB] Connection to 'production' closed

  [DB] Connected to 'production'
  Permission denied: DROP not allowed!
  [DB] Connection to 'production' closed
